# 04v2 - A3: integrate the published slot cache

**Plan item A3** ([[project-rsna-phase-status]] / the strategy artifact):
rebuild preprocessing cleanly, without reconstructing the lost
`05a_weak_dicom_preprocess.ipynb`.

**Decision (2026-08-26, user):** given this project's own preprocessing
track record (Fase 4's laterality-flip/metric-leak/normalization bugs,
A0b's slice-ordering bug on 71% of studies, `06b`'s unexplained
intensity residual), **reuse a published, already-built cache instead of
rebuilding our own DICOM pipeline** — see
[[feedback-prefer-reuse-over-rebuild-preprocessing]].

**Source, confirmed by reading the actual pipeline code** (not just the
forum post): `stevenleehans/rsna-knee-500gb-to-11gib-cpu-pixel-cache`
(Kaggle notebook, downloaded to
`data/raw/_reference_kernels/rsna-knee-500gb-to-11gib-cpu-pixel-cache.ipynb`,
same convention as the other reference kernels there). Its `Output` is an attachable Kaggle Dataset: 4
train shards + 3 test shards, each with `_cache.npy` (pixel data),
`_mask.npy` (slot-presence), `_studies.csv` (row -> `StudyInstanceUID`
key), plus one `cache_meta.json` documenting the build config.

**What the cache actually contains** (from `cache_meta.json` + the
source code's `SLOTS_RECOVERED`, `read_slot`, `take_group`):
- Per study: **6 named slots** (`SAG_FLUID_FS`, `COR_FLUID_FS`,
  `AX_FLUID_FS`, `SAG_FLUID_NOFS`, `COR_T1`, `SAG_T1`) — plane crossed
  with fluid-sensitivity and fat-suppression **recovered independently
  from the raw DICOM header**, not from `train_series.csv`'s two columns
  directly (the source's own EDA found `Fluid_Sensitive`/
  `Fat_Suppression` are byte-identical across all 24,371 series rows —
  independently reproduced by us locally against `train_series.csv`,
  see [[project-rsna-phase-status]]).
- Per slot: **9 slices** = 3 "anchors" spread across a sampling window,
  each anchor contributing 3 physically-adjacent slices
  (`group=3, n_group=3`). `take_group(rows, g) = rows[:, :, g*3:(g+1)*3]`
  — group index 0/1/2 are the low/centre/high anchors in that order
  (`np.linspace(lo, hi, 3)` is ascending), so **group index 1 is the
  centre anchor**.
- `crop_mm=130` (physical crop, not the historical 160 — chosen because
  160 sits at this corpus's measured median FOV, which silently disabled
  the crop on ~61% of studies), `img=224`, `uint8`.
- **Important nuance on the "3 vs 9 slices" question** (the user's
  instruction was to follow the forum's own result, which favoured 3
  centre slices over 9): the forum's own 3-vs-9 comparison (`exp-016`,
  +0.0086 AUC, 4x their noise floor) used a *wide* sampling window
  (10-90% / 15-85% of the stack per plane), so the 3 anchors were spread
  to the window's edges — "fewer slices" and "more central slices" were
  **confounded** in that result, flagged by the source itself. **This
  specific cache build sets `RSNA_WINDOW=0.35,0.65`**, a narrow window,
  *specifically to remove that confound* — quoting the source: "RSNA_WINDOW
  narrows the window without changing the count, which separates them."
  Net effect: in this cache, all 3 anchor groups are already close to
  centre (not spread to wide edges), so the original "9 is worse because
  it reaches the edges" argument doesn't automatically transfer here. We
  don't have a re-measured 3-vs-9 result for this narrower-window build
  (the published notebook ships with `RSNA_FOLDS=1`, a single-fold proof
  run, not a full report). **Practical resolution below: keep all 9
  slices cached (already true — nothing to redo) and make the group
  selection a load-time parameter, defaulting to group 1 (centre) to
  honour the forum's steer, but cheap to change to "all 3 groups" and
  re-validate at A2 against our own CV — no Kaggle re-run needed either
  way.**

**This notebook needs the published cache Dataset attached to a Kaggle
kernel** (multi-GB `.npy` files, not something to download locally) —
run on Kaggle and report back the real output, same convention as
`00v2`/`03v2`.

## Setup

`CACHE_DIR` is hardcoded to the user's own re-upload of the cache
Dataset, same convention as `src/config.py::_KAGGLE_INPUT_DIR` for the
competition data itself - this project doesn't auto-discover Kaggle
input paths anywhere else, no reason to start here.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

_KAGGLE_RAW = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
ON_KAGGLE = _KAGGLE_RAW.exists()
if not ON_KAGGLE:
    raise RuntimeError(
        "This notebook needs the published cache Dataset attached to a "
        "Kaggle kernel (multi-GB .npy files) - run on Kaggle, not locally."
    )

REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

RAW_DIR = _KAGGLE_RAW

CACHE_DIR = Path("/kaggle/input/datasets/alherma7/cache-stevenleehans-rsna/cache")
if not (CACHE_DIR / "cache_meta.json").exists():
    raise RuntimeError(
        f"{CACHE_DIR} has no cache_meta.json - check the Dataset is "
        "attached to this kernel and CACHE_DIR still matches its mount path."
    )
print("CACHE_DIR:", CACHE_DIR)

## `cache_meta.json` - confirm the build config matches what we read in the source notebook

In [ ]:
with open(CACHE_DIR / "cache_meta.json") as f:
    meta = json.load(f)
print(json.dumps(meta, indent=2))

EXPECTED_SLOTS = [
    "SAG_FLUID_FS", "COR_FLUID_FS", "AX_FLUID_FS",
    "SAG_FLUID_NOFS", "COR_T1", "SAG_T1",
]
assert meta["slots"] == EXPECTED_SLOTS, meta["slots"]
assert meta["crop_mm"] == 130.0, meta["crop_mm"]
assert meta["img"] == 224, meta["img"]
assert meta["group"] == 3, meta["group"]
assert meta["n_group"] == 3, meta["n_group"]
assert meta["shards"] == 4, meta["shards"]
assert meta["window"] == "0.35,0.65", meta["window"]
print("\nmeta matches the source code read on 2026-08-26 - OK")

## One shard - confirm array shapes line up with the meta

In [ ]:
shard = "train.s00of04"
cache = np.load(CACHE_DIR / f"{shard}_cache.npy", mmap_mode="r")
mask = np.load(CACHE_DIR / f"{shard}_mask.npy", mmap_mode="r")
studies = pd.read_csv(CACHE_DIR / f"{shard}_studies.csv")

print("cache:", cache.shape, cache.dtype)
print("mask:", mask.shape, mask.dtype)
print("studies:", studies.shape)

n = meta["splits"][shard]["studies"]
assert cache.shape == (n, 6, 9, 224, 224), cache.shape
assert mask.shape == (n, 6), mask.shape
assert len(studies) == n, len(studies)
print(f"\n{shard}: {n} studies, shapes consistent - OK")

## All 4 train shards - coverage vs. the full corpus

Reads only `_studies.csv` and `_mask.npy` here (small files) - no need
to touch the multi-GB `_cache.npy` arrays for a coverage check.

In [ ]:
all_studies = []
all_masks = []
for i in range(4):
    shard = f"train.s{i:02d}of04"
    s = pd.read_csv(CACHE_DIR / f"{shard}_studies.csv")
    m = np.load(CACHE_DIR / f"{shard}_mask.npy")
    all_studies.append(s)
    all_masks.append(m)

studies_all = pd.concat(all_studies, ignore_index=True)
mask_all = np.concatenate(all_masks, axis=0)
assert len(studies_all) == len(mask_all)

print("total studies across 4 shards:", len(studies_all))
print("unique StudyInstanceUID:", studies_all["StudyInstanceUID"].nunique())
assert studies_all["StudyInstanceUID"].nunique() == len(studies_all), "duplicate studies across shards"

train_csv = pd.read_csv(RAW_DIR / "train.csv")
known_ids = set(train_csv["StudyInstanceUID"])
cache_ids = set(studies_all["StudyInstanceUID"])
print("cache studies not in train.csv:", len(cache_ids - known_ids))
print("train.csv studies missing from cache:", len(known_ids - cache_ids))
assert cache_ids <= known_ids, "cache has StudyInstanceUIDs train.csv doesn't recognise"

label_cols = [c for c in train_csv.columns if c not in ("StudyInstanceUID", "Report")]
gold_mask = train_csv[label_cols].notna().all(axis=1)
gold_ids = set(train_csv.loc[gold_mask, "StudyInstanceUID"])
print("gold studies (58 expected):", len(gold_ids))
print("gold studies present in cache:", len(gold_ids & cache_ids))
assert gold_ids <= cache_ids, "some gold studies are missing from the cache"
print("\nAll gold studies are covered by the cache - OK")

## Slot presence - cross-check against our own `train_series.csv` count

Not an exact match (the cache's slot recovery reads raw DICOM headers;
our local check only has the two `train_series.csv` columns), just a
same-ballpark sanity check.

In [ ]:
avg_slots_cache = mask_all.sum(axis=1).mean()
print(f"avg slots present per study (cache mask): {avg_slots_cache:.2f} / 6")

train_series = pd.read_csv(RAW_DIR / "train_series.csv")
train_series["slot"] = train_series["Anatomical_Plane"] + "_" + train_series["Fluid_Sensitive"].astype(str)
avg_slots_local = train_series.groupby("StudyInstanceUID")["slot"].nunique().mean()
print(f"avg distinct (plane, fluid-sensitivity) combos per study (train_series.csv): {avg_slots_local:.2f} / 6")

all_six_cache = (mask_all.sum(axis=1) == 6).mean()
print(f"studies with all 6 slots present (cache): {all_six_cache:.1%}")

## Visual sanity check

A handful of studies with all 6 slots present, group index 1 (centre
anchor, 3 channels) per slot, as a grayscale grid. Confirms the cached
pixels are real, correctly-oriented knee MRI and not corrupted/blank -
a qualitative check, not just shapes matching.

Uses `cache`/`mask`/`studies` from the single-shard cell above (shard
`train.s00of04` only, already loaded) rather than the 4-shard
concatenation - `mask_all`/`studies_all` index the full 4,407-study
space but `cache` only holds shard 0's rows, so mixing the two would
index out of range.

In [ ]:
import matplotlib.pyplot as plt

SLOT_NAMES = meta["slots"]
GROUP = 1  # centre anchor, see the markdown discussion above

full_idx = np.flatnonzero(mask.sum(axis=1) == 6)
rng = np.random.default_rng(42)
sample_idx = rng.choice(full_idx, size=min(3, len(full_idx)), replace=False)

fig, axes = plt.subplots(len(sample_idx), 6, figsize=(18, 3 * len(sample_idx)))
if len(sample_idx) == 1:
    axes = axes[None, :]

for row, idx in enumerate(sample_idx):
    study_id = studies.iloc[idx]["StudyInstanceUID"]
    for col, slot_name in enumerate(SLOT_NAMES):
        img = cache[idx, col, GROUP * 3 + 1]  # middle channel of the centre group
        axes[row, col].imshow(img, cmap="gray")
        axes[row, col].set_title(slot_name, fontsize=9)
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(study_id[-8:], fontsize=8)

plt.tight_layout()
plt.savefig("sample_slot_cache_grid.png", dpi=100)
plt.show()
print("saved sample_slot_cache_grid.png")

## Group-selection helper

`select_group` below is the candidate for `src/data.py` (or
`src/features.py`) once this notebook is validated end to end. Mirrors
the source's own `take_group` indexing exactly - same slicing, same
group-index meaning (0/1/2 = low/centre/high anchor).

In [ ]:
def select_group(cache_slot_stack, group_index):
    '''Select one or more anchor groups from a (..., 9, H, W) slot stack.

    `group_index`: int (one group, 3 channels) or a sequence of ints
    (multiple groups, concatenated on the channel axis). Group 0/1/2 are
    the low/centre/high sampling anchors, in that order - matches
    stevenleehans's rsna-knee-500gb-to-11gib-cpu-pixel-cache::take_group.
    '''
    if isinstance(group_index, int):
        group_index = [group_index]
    chans = [cache_slot_stack[..., g * 3:(g + 1) * 3, :, :] for g in group_index]
    return np.concatenate(chans, axis=-3)


# Demonstrate on one study, one slot: centre group only vs. all 3 groups.
demo = cache[sample_idx[0], 0]  # (9, 224, 224)
centre_only = select_group(demo, 1)
all_groups = select_group(demo, [0, 1, 2])
print("centre-only shape:", centre_only.shape)   # (3, 224, 224)
print("all-groups shape:", all_groups.shape)     # (9, 224, 224)
assert centre_only.shape == (3, 224, 224)
assert all_groups.shape == (9, 224, 224)
assert np.array_equal(all_groups[3:6], centre_only)
print("select_group matches take_group's indexing - OK")

## Real output (run 2026-08-26, on Kaggle, cache Dataset attached)

**Meta check:** `cache_meta.json` matched the source-code-derived
expectations exactly (`crop_mm=130`, `img=224`, `group=3`, `n_group=3`,
`shards=4`, `window="0.35,0.65"`, the 6 named slots) - OK.

**Shapes:** `cache: (1101, 6, 9, 224, 224) uint8`,
`mask: (1101, 6) float32`, `studies: (1101, 1)` for shard `train.s00of04`
- matches `meta["splits"]` exactly - OK.

**Coverage, all 4 train shards:** 4,407 total studies, 4,407 unique
`StudyInstanceUID` (no cross-shard duplicates), 0 cache studies missing
from `train.csv`, 0 `train.csv` studies missing from the cache, and all
**58/58 gold studies present** - OK.

**Slot-presence cross-check - real, explained discrepancy, not a bug:**
cache mask says 5.24/6 slots present on average (47.2% of studies have
all 6); our own local estimate from `train_series.csv` (a naive
symmetric "3 planes x 2 fluid-sensitivity" grid) said 4.84/6 (13% with
all 6). The two numbers answer different questions, not a contradiction:
the cache's 6 named slots are **not** that symmetric grid - there is no
`AX` non-fluid slot and no `COR` fluid-without-fat-sat slot in
`SLOTS_RECOVERED` at all (dropped because those combinations are rare in
this corpus), so a study is far more likely to satisfy the curated
6-slot set than the naive grid, which counts rare/near-nonexistent
combinations against every study.

**`select_group()`:** `centre_only.shape == (3, 224, 224)`,
`all_groups.shape == (9, 224, 224)`, and `all_groups[3:6]` equals
`centre_only` exactly - matches the source's `take_group` indexing - OK.

**Visual check:** `sample_slot_cache_grid.png` (3 studies x 6 slots,
group-1/centre-anchor middle channel, inspected 2026-08-26) shows
correctly-oriented, recognisable knee MRI for every slot in every
sampled study - sagittal/coronal views show the expected femur-tibia
joint anatomy, axial views show the expected cross-sectional joint
shape, fat-suppressed vs. non-fat-suppressed and T1 vs. fluid-sensitive
contrast look physically distinct as expected, no blank/corrupted/
mislabelled images. **The image itself is not committed to this
repo** - it is real patient MRI data, same policy as
`sample_triplets.png` (see `.gitignore`).

**Conclusion: A3's cache-integration notebook is fully validated.**
`select_group()` is ready to graduate to `src/data.py` with a unit test
(synthetic small arrays, no real Kaggle data needed for the test
itself), pending the user's go-ahead per
[[feedback-user-validates-notebook-before-src]].